In [1]:
import cv2
import numpy as np
import tensorflow as tf
import os
import paho.mqtt.client as mqtt
import time

In [2]:
# --- MQTT CONFIG ---
BROKER = "broker.hivemq.com"  # ganti kalo ada broker lokal
PORT = 1883
TOPIC = "sic/dibimbing/DoaIbuMenyertai/cam"

client = mqtt.Client()
client.connect(BROKER, PORT, 60)
print(f"✅ MQTT connected to {BROKER}:{PORT}, topic: {TOPIC}")

C:\Users\Marcell\AppData\Local\Temp\ipykernel_7184\3684859346.py:6: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


✅ MQTT connected to broker.hivemq.com:1883, topic: sic/dibimbing/DoaIbuMenyertai/cam


In [3]:
# --- LOAD MODEL ---
interpreter = tf.lite.Interpreter(model_path="vww_96_grayscale_quantized.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input shape:", input_details[0]['shape'])

labels = ["Fokus", "Gak Fokus"]
try:
    with open("labels.txt", "r") as f:
        labels = [line.strip() for line in f.readlines()]
except FileNotFoundError:
    pass

Input shape: [ 1 96 96  1]


c:\Users\Marcell\anaconda3\envs\iot\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# --- CAMERA STREAM ---
cap = cv2.VideoCapture("http://192.168.18.82:81/stream")
if not cap.isOpened():
    print("❌ Gagal buka webcam")
    exit()

print("✅ Webcam aktif! Tekan 'q' untuk keluar.")

last_sent = 0  # biar gak spam publish

In [10]:
while True:
    ret, frame = cap.read()
    if not ret:
        print("Frame gagal")
        break
    
    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame = cv2.equalizeHist(gray)
    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

    # Grayscale & resize sesuai model
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    input_shape = input_details[0]['shape']
    h, w = input_shape[1], input_shape[2]
    img = cv2.resize(frame_gray, (w, h))
    img = np.expand_dims(img.astype(np.float32) / 255.0, axis=(0, -1))

    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]['index'])
    pred_idx = int(np.argmax(output_data))
    confidence = float(np.max(output_data))

    label = f"{labels[pred_idx]} ({confidence*100:.1f}%)"
    color = (0, 255, 0) if pred_idx == 0 else (0, 0, 255)
    H, W, _ = frame.shape
    font_scale = min(W, H) / 500
    cv2.putText(frame, label, (10, int(H * 0.1)), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 1)

    # --- MQTT Publish (setiap 2 detik biar gak spam) ---
    now = time.time()
    if now - last_sent > 2:
        message = {
            "status": labels[pred_idx],
            "confidence": round(confidence, 3)
        }
        client.publish(TOPIC, str(message))
        print(f"📡 Published: {message}")
        last_sent = now

    cv2.imshow("Focus Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
client.disconnect()
print("🚪 Closed all connections.")

📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.975}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.964}
📡 Published: {'status': '0 Fokus', 'confidence': 0.875}
📡 Published: {'status': '0 Fokus', 'confidence': 0.958}
📡 Published: {'status': '0 Fokus', 'confidence': 0.962}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.864}
📡 Published: {'status': '0 Fokus', 'confidence': 0.95}
📡 Published: {'status': '0 Fokus', 'confidence': 0.678}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.653}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.63}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.754}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.723}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.809}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.921}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.976}
📡 Published: {'status': '1 Kagak FouKu5', 'confidence': 0.987}
📡 Published: {'status': '1 Ka